# CSV Utilities (`csv_utilities`)

El módulo `csv_utilities` (`ds_utils.io.csv_utilities`) proporciona funciones para importar y exportar DataFrames mediante archivos CSV.

- **`export_csv`**: exporta un DataFrame a un archivo CSV y crea automáticamente los directorios de destino si no existen.
- **`import_csv`**: importa un archivo CSV a un DataFrame de Pandas, con opciones para mostrar una previsualización y generar una definición de contrato.

Este notebook muestra un ejemplo básico de exportación e importación de datos, y elimina los archivos generados al finalizar.

In [24]:
# AUTOMATIC RELOAD OF EXTERNAL MODULES
%load_ext autoreload
%autoreload 2

from pathlib import Path

from tempfile import TemporaryDirectory

import pandas as pd

from ds_utils.io.csv_utilities import export_csv, import_csv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Create a sample DataFrame

Creamos un DataFrame de ejemplo con datos de sensores para comprobar que los datos se conservan después de exportarlos e importarlos.

In [25]:
df_original = pd.DataFrame({
    "timestamp": pd.date_range("2025-01-01", periods=5, freq="h"),
    "temperature": [21.5, 22.1, 20.8, 21.9, 22.4],
    "sensor": ["A", "B", "A", "B", "A"],
})

display(df_original)

,timestamp,temperature,sensor
0,2025-01-01 00:00:00,21.5,A
1,2025-01-01 01:00:00,22.1,B
2,2025-01-01 02:00:00,20.8,A
3,2025-01-01 03:00:00,21.9,B
4,2025-01-01 04:00:00,22.4,A


## 2. Create a Temporary Directory

Crea un directorio temporal para almacenar el archivo CSV generado durante este ejemplo. El directorio es gestionado por Python y puede eliminarse cuando ya no sea necesario.
La ruta del directorio se almacena en `temp_path`, mientras que `csv_path` define la ubicación del archivo CSV que se exportará e importará en los pasos siguientes.
El directorio temporal se eliminará explícitamente al finalizar el ejemplo mediante `temp_dir.cleanup()`.

In [26]:
temp_dir = TemporaryDirectory()
temp_path = Path(temp_dir.name)

csv_path = temp_path / "sensors.csv"

print(f"Temporary directory: {temp_path}")

Temporary directory: /tmp/tmp40hp9c0m


## 3. Export a DataFrame to CSV

Utilizamos `export_csv` para guardar el DataFrame. El directorio de destino es temporal y se eliminara al finalizar.

Se utiliza `index=False` para evitar guardar el índice del DataFrame como una columna adicional.

In [27]:
export_csv(
    df_original,
    csv_path,
    index=False,
)

print(f"CSV exported successfully to: {csv_path.resolve()}")

CSV exported successfully to: /tmp/tmp40hp9c0m/sensors.csv


## 4. Import a DataFrame from CSV

Leemos el archivo generado mediante `import_csv` y mostramos su contenido.
Usamos `parse_dates` para convertir la columna 'timestamp' de tipo `object` a `datetime64[ns]`

In [28]:
df_imported = import_csv(
    csv_path,
    parse_dates=["timestamp"], # cambiar timestamp de tipo `object` a `datetime64[ns]`
)

display(df_imported)

,timestamp,temperature,sensor
0,2025-01-01 00:00:00,21.5,A
1,2025-01-01 01:00:00,22.1,B
2,2025-01-01 02:00:00,20.8,A
3,2025-01-01 03:00:00,21.9,B
4,2025-01-01 04:00:00,22.4,A


## 5. Validate the result

Comprobamos que el DataFrame importado coincide con el original.

In [21]:
pd.testing.assert_frame_equal(df_original, df_imported)

print("CSV export and import validated successfully.")

CSV export and import validated successfully.


## 6. Clean up generated files

Eliminamos la carpeta temporal creada.

In [30]:
temp_dir.cleanup()

# Verify that the temporary file and directory have been removed.
print(f"CSV file successfully deleted: {not csv_path.exists()}")
print(f"Temporary directory successfully deleted: {not temp_path.exists()}")

CSV file successfully deleted: True
Temporary directory successfully deleted: True
